# Component 2 — Does the Buy Now / Wait advice save money? (cost simulation)

**Research question.** If a shop owner followed the procurement advice during the test period,
how much would they save in LKR, compared with what they do today and with simple rules?

ROC-AUC and F1 (in `procument_timesplit.ipynb`) say how often the advice is *right*. A shop owner
cares about **money**: a wrong "wait" on an item whose price jumps costs more than a wrong "wait"
on an item whose price barely moves. This notebook turns every decision into a purchase cost.

**Simulation.** For every item and every test week (2025-08-25 → 2026-06-15, never used for training):

- **Buy now** → pay today's price `current_price_rs`.
- **Wait** → pay the actual price 4 weeks later `expected_price_next_4wk_rs`.
- One unit (1 kg / 1 item) per decision, so items count equally.

| | Policy | Decides "buy now" when |
|---|---|---|
| 0 | **Always buy now** | always — what a shop does today without advice (the reference) |
| 1 | Rule: price fell | the price fell over the last 4 weeks |
| 2 | Rule: below average | today's price is below its 3-month average |
| 3 | ML classifier | the Buy/Wait model says P(price will rise) ≥ 0.5 |
| 4 | ML price forecast | the forecast price in 4 weeks is higher than today's price |
| ★ | Perfect foresight | the future price is higher (an upper bound no method can beat) |

Models are trained exactly as in `procument_timesplit.ipynb` (development period only, 4-week gap).
The original notebooks, dataset and `.pkl` files are not changed, and nothing is saved over them.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## 1. Same data, time order and split as `procument_timesplit.ipynb`

In [ ]:
df = pd.read_csv('from_rice_veg_2023_2026.csv').drop_duplicates().reset_index(drop=True)
df['week_start'] = pd.to_datetime(
    df['iso_year'].astype(str) + '-W' + df['iso_week'].astype(str).str.zfill(2) + '-1', format='%G-W%V-%u')
df = df.sort_values(['week_start', 'item']).reset_index(drop=True)

X = df.drop(columns=['target_buy_now', 'expected_price_next_4wk_rs', 'week_start'])
y_cls = df['target_buy_now'].astype(int)
y_reg = df['expected_price_next_4wk_rs'].astype(float)
cat_cols = ['item', 'category']
num_cols = [c for c in X.columns if c not in cat_cols]

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ])

GAP = pd.Timedelta(weeks=4)
TEST_START = pd.Timestamp('2025-01-01')
dev = (df['week_start'] < TEST_START - GAP).values
test = (df['week_start'] >= TEST_START).values
print(f'Development: {dev.sum()} rows | Test: {test.sum()} rows, '
      f"{df.loc[test, 'week_start'].min().date()} -> {df.loc[test, 'week_start'].max().date()}, "
      f"{df.loc[test, 'item'].nunique()} items")

## 2. Train the models on the past only

Same champion and settings as the time-split notebook (RandomForest classifier, RandomForest price regressor).

In [ ]:
classifier = Pipeline([('prep', make_preprocessor()), ('clf', RandomForestClassifier(
    n_estimators=300, max_depth=10, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
regressor = Pipeline([('prep', make_preprocessor()), ('reg', RandomForestRegressor(
    n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1))])
classifier.fit(X[dev], y_cls[dev])
regressor.fit(X[dev], y_reg[dev])

T = df[test].copy()
XT = X[test]
T['p_rise'] = classifier.predict_proba(XT)[:, 1]
T['forecast_4wk'] = regressor.predict(XT)

## 3. The decisions of each policy and what they cost

In [ ]:
now, later = T['current_price_rs'], T['expected_price_next_4wk_rs']
policies = {
    '0. Always buy now (today)':   np.ones(len(T), dtype=bool),
    '1. Rule: price fell (4 wk)':  (T['price_change_4wk_pct'].fillna(0) < 0).to_numpy(),
    '2. Rule: below 3-mo average': (T['price_vs_3mo_avg_pct'].fillna(0) < 0).to_numpy(),
    '3. ML classifier':            (T['p_rise'] >= 0.5).to_numpy(),
    '4. ML price forecast':        (T['forecast_4wk'] > now).to_numpy(),
    '★ Perfect foresight':         (later > now).to_numpy(),
}
cost = {name: np.where(buy, now, later) for name, buy in policies.items()}

ref = cost['0. Always buy now (today)'].sum()
best = ref - cost['★ Perfect foresight'].sum()
rows = []
for name, c in cost.items():
    saved = ref - c.sum()
    rows.append({'policy': name,
                 'total spend (LKR)': c.sum(),
                 'saved vs today (LKR)': saved,
                 'saved %': saved / ref * 100,
                 '% of possible saving': saved / best * 100,
                 'buy-now share %': policies[name].mean() * 100})
summary_raw = pd.DataFrame(rows).set_index('policy')
summary = summary_raw.round(1)
print(f'{len(T)} purchase decisions, 1 unit each. Spend if always buying now: LKR {ref:,.0f}')
summary

**How to read it.** *saved %* is the saving on the whole purchase bill. *% of possible saving* compares
with perfect foresight: 100 % would mean every decision was the cheapest one possible.
A negative saving means the policy is **worse** than simply buying now.

## 4. Is ML's saving really bigger than the rule's? Bootstrap by week

Decisions in the same week share market conditions, so whole **weeks** are resampled (1,000 times).
If the 95 % interval of the difference does not include 0, the difference is unlikely to be luck.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
weeks = T['week_start'].to_numpy()
uniq = np.unique(weeks)
week_rows = [np.where(weeks == w)[0] for w in uniq]
saved_rows = {name: cost['0. Always buy now (today)'] - c for name, c in cost.items()}

def boot_saving_pct(name):
    out = []
    for _ in range(1000):
        pick = np.concatenate([week_rows[i] for i in rng.integers(0, len(uniq), len(uniq))])
        out.append(saved_rows[name][pick].sum() / cost['0. Always buy now (today)'][pick].sum() * 100)
    return np.array(out)

boot = {name: boot_saving_pct(name) for name in ['1. Rule: price fell (4 wk)', '3. ML classifier', '4. ML price forecast']}
for name, b in boot.items():
    lo, hi = np.percentile(b, [2.5, 97.5]); print(f'{name:28s} saving {b.mean():5.2f}%  95% CI [{lo:5.2f}, {hi:5.2f}]')
print()
for ml in ['3. ML classifier', '4. ML price forecast']:
    d = boot[ml] - boot['1. Rule: price fell (4 wk)']
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f'{ml} minus rule: {d.mean():+.2f} percentage points  95% CI [{lo:+.2f}, {hi:+.2f}]')

## 5. Where does the saving come from? By category

In [ ]:
by_cat = pd.DataFrame({
    'category': T['category'].to_numpy(),
    'spend today': cost['0. Always buy now (today)'],
    'saved: rule': saved_rows['1. Rule: price fell (4 wk)'],
    'saved: ML forecast': saved_rows['4. ML price forecast'],
    'saved: perfect': saved_rows['★ Perfect foresight'],
}).groupby('category').sum()
for c in ['saved: rule', 'saved: ML forecast', 'saved: perfect']:
    by_cat[c.replace('saved', 'saved %')] = by_cat[c] / by_cat['spend today'] * 100
by_cat.round(1)

In [ ]:
order = [p for p in policies if p != '0. Always buy now (today)']
s = summary_raw.loc[order]
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#999', '#999', '#6a9fd4', '#1f5fa8', '#2e7d32']
ax.barh(s.index, s['saved %'], color=colors)
ax.axvline(0, color='black', lw=0.8)
for i, v in enumerate(s['saved %']):
    ax.text(v + (0.05 if v >= 0 else -0.05), i, f'{v:.2f}%', va='center', ha='left' if v >= 0 else 'right', fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Saving on the purchase bill vs always buying now (%)')
ax.set_title('Test period 2025-08 to 2026-06: money saved by each policy')
plt.tight_layout(); plt.savefig('component2_savings_simulation.png', dpi=120); plt.show()

## 6. Conclusion

*Written after reading the numbers above — fill in with the actual results:*

- Saving with the ML advice: ___ % of the purchase bill (LKR ___ over ___ decisions).
- Compared with the simple rule: ___ (bootstrap interval ___).
- Share of the best possible saving captured: ___ %. Category where it helps most: ___.

**Assumptions and limitations.**
- One unit per decision; real shops buy different quantities per item.
- "Wait" assumes the shop has enough stock to wait 4 weeks (in the app, the advice is shown together with stock level).
- Each decision is treated separately; storage cost, spoilage and transport are ignored.
- The price series is built from public market price data, not from one real shop's purchases.